<a href="https://colab.research.google.com/github/Aniketh78/Generative-AI-Lab_Experiments/blob/main/GenAI_EXP_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import google.generativeai as genai
import requests
import time
from IPython.display import display, Markdown

GEMINI_API_KEY = "API_KEY_IS_HIDDEN"
HF_API_KEY = "SAME_HERE"

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-3-flash-preview")

HF_MODEL = "meta-llama/Llama-3.1-8B-Instruct:cerebras"
HF_URL = "https://router.huggingface.co/v1/chat/completions"
hf_headers = {
    "Authorization": f"Bearer {HF_API_KEY}",
    "Content-Type": "application/json"
}

def query_gemini(prompt):
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"Gemini Error: {e}"

def query_huggingface(prompt):
    payload = {
        "model": HF_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 80,
        "temperature": 0.7
    }

    for _ in range(3):
        try:
            response = requests.post(HF_URL, headers=hf_headers, json=payload)

            if response.status_code == 200:
                data = response.json()
                return data["choices"][0]["message"]["content"].strip()

            elif response.status_code == 503:
                time.sleep(5)

            else:
                return f"HF Error {response.status_code}: {response.text}"

        except Exception as e:
            return f"HF Exception: {e}"

    return "HF model loading timeout."

prompt = "Write a haiku about AI and nature"

gemini_output = query_gemini(prompt)
hf_output = query_huggingface(prompt)

print("\nPaste OpenAI response below:\n")
openai_output = input()

display(Markdown("Model Responses"))

display(Markdown("Gemini"))
display(Markdown(gemini_output))

display(Markdown("Hugging Face (Llama-3.1-8B)"))
display(Markdown(hf_output))

display(Markdown("OpenAI (Manual)"))
display(Markdown(openai_output))

def evaluate(prompt, responses):
    eval_prompt = f"""
You are an evaluator.

Prompt:
{prompt}

Responses:
1. OpenAI: {responses['OpenAI']}
2. Gemini: {responses['Gemini']}
3. HuggingFace: {responses['HuggingFace']}

Evaluate based on:
- Creativity
- Relevance
- Fluency

Return ONLY ranking:
1. ModelName
2. ModelName
3. ModelName
"""
    try:
        result = model.generate_content(eval_prompt)
        return result.text.strip()
    except Exception as e:
        return f"Evaluation Error: {e}"

responses = {
    "OpenAI": openai_output,
    "Gemini": gemini_output,
    "HuggingFace": hf_output
}

ranking = evaluate(prompt, responses)

display(Markdown("Ranking (Best to Worst)"))
display(Markdown(ranking))


Paste OpenAI response below:

Circuits hum like wind, Roots of code in forest soil, Thoughts bloom—silicon.


Model Responses

Gemini

Silicon and leaf,
Logic flows like mountain streams,
Green code starts to bloom.

Hugging Face (Llama-3.1-8B)

Metal wings unfold
Dancing with the morning dew
Art meets gentle breeze

OpenAI (Manual)

Circuits hum like wind, Roots of code in forest soil, Thoughts bloom—silicon.

Ranking (Best to Worst)

1. OpenAI
2. Gemini
3. HuggingFace